# OpenSim Scale and IK Template

Use this notebook after you have a TRC file and a compatible OpenSim model. Run scale first, inspect the output, then run inverse kinematics.

## 1. Import and set paths

In [ ]:
from pathlib import Path
import monomech as mm

TRC_PATH = Path("outputs/subject01/subject01_global.trc")
OUTPUT_DIR = Path("outputs/subject01")
MODEL_PATH = mm.get_builtin_osim_model("pose")

print("TRC:", TRC_PATH)
print("Model:", MODEL_PATH)

## 2. Load the TRC as a marker trial

This gives you a trial object that can run OpenSim helpers and summarize marker data.

In [ ]:
trial = mm.load_trc(TRC_PATH)
display(trial.summary().head(20))

## 3. Check marker names against the model

In [ ]:
marker_check = trial.validate_against_model(MODEL_PATH)
display(marker_check.head(30))
print("Markers missing from model:", int((~marker_check["in_model"]).sum()))

## 4. Run scale

For real analysis, choose a quiet standing time range when possible. Leave `start_time` and `end_time` unset to use the available TRC range.

In [ ]:
scale = trial.run_opensim_scale(
    model_path=MODEL_PATH,
    trc_path=TRC_PATH,
    output_dir=OUTPUT_DIR / "scale",
    # start_time=0.0,
    # end_time=1.0,
)

display(scale.summary())

## 5. Run inverse kinematics

In [ ]:
ik = trial.run_opensim_ik(
    model_path=scale.scaled_model_path,
    trc_path=TRC_PATH,
    output_dir=OUTPUT_DIR / "ik",
)

print("IK MOT:", ik.mot_path)
display(ik.summary())

## 6. Optional inverse dynamics

Inverse dynamics requires a reasonable IK result and external force information. Add external loads only after checking that scale and IK outputs are sound.

In [ ]:
# id_result = trial.run_opensim_id(
#     model_path=scale.scaled_model_path,
#     ik_path=ik.mot_path,
#     output_dir=OUTPUT_DIR / "id",
# )
# display(id_result.summary())